# GUI Build
This is a test build of the GUI before we successfully integrated the spectrometer. It uses data in the same form as the spectrometer generates for testing purposes, but does not link to the spectrometer yet.

### Some notes:
- once the spectrometer is linked, the thread running _live_loop may need to be "inverted" (since the delay will happen after requesting spectrometer data)
- the file saving system still needs to be worked out
- eventually, the GUI class should be linked to a Spectrometer from seabreeze

In [2]:
import matplotlib
matplotlib.use('TkAgg')
import time
import threading 
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

%matplotlib widget

rng = np.random.default_rng()

def noisyGaussian(x):
    output = np.zeros_like(x)
    center = np.mean(x)
    width = np.std(x)/2
    output += np.exp( -((x-center)/width)**2  )
    output += rng.random(len(x))*0.1
    return output 

xx = np.linspace(250, 800, 1024)

class PL_GUI:
    def __init__(self, xx):
        self.xx = xx
        self.plot_output = widgets.Output() 
        self.text_output = widgets.Output() 
        
        # Initalize variables
        self.live = False #tracks if the feed is live
        self.loop_thread = None  # Tracks if thread is running

        # Create inputs
        self.exposure = widgets.FloatText(description='Exposure (ms)', value=200)
        self.noOfAv_input = widgets.IntText(description='Averages', value=1)
        self.modeSelector = widgets.RadioButtons(
            options=['Single Spectrums', 'Live View'], 
            description='GUI mode'
        )
        self.saveFig = widgets.Button(description='Save current figure')

        # Link widgets to functions
        self.modeSelector.observe(self.onModeSelection, names='value')
        self.exposure.observe(self.onExposureUpdate, names='value')
        self.noOfAv_input.observe(self.onAvUpdate, names='value')
        self.saveFig.on_click(self.onFigSaveRequested)

        #Intialize variables to store values
        self.exposureTime = self.exposure.value
        self.noOfAv = self.noOfAv_input.value

        # Assemble the layout
        self.layout = widgets.HBox([ 
            self.plot_output,
            widgets.VBox([
                self.exposure,
                self.noOfAv_input,
                self.modeSelector,
                self.saveFig,
                self.text_output
            ])
        ])

        # Generate the plot
        with self.plot_output:
            plt.close('all') 
            self.fig, self.ax = plt.subplots(figsize=(6, 4))
            self.ax.set_xlabel('Wavelength (nm)')
            self.ax.set_ylabel('Intensity (a.u.)')
            self.ax.set_ylim(-0.1, 1.2)
            self.data, = self.ax.plot(self.xx, noisyGaussian(self.xx), label='Live')
            self.ax.legend()
            plt.show(self.fig)

    def _ipython_display_(self):
        '''Ensures the GUI is displayed using ipython'''
        display(self.layout)

    def onModeSelection(self, change):
        '''Runs when the "mode" radio button is changed.'''
        if change['new'] == 'Single Spectrums':
            self.stopLiveFeed()
        elif change['new'] == 'Live View':
            self.startLiveFeed()

    def startLiveFeed(self):
        '''Initalizes a thread to process data in the background and push to GUI when done'''
        self.stopLiveFeed() #kill any prior 
        if not self.live:
            self.live = True
            self.loop_thread = threading.Thread(target=self._live_loop, daemon=True) #assing a thread to the _live_loop func
            self.loop_thread.start() #and start it

    def stopLiveFeed(self):
        self.live = False #we do not need to join() thread, we will let it kill itself when 

    def _live_loop(self):
        '''The loop that handles requesting data from the spectrometer and pushes it to the GUI.
        Currently, it only simulates data like the spectrometer.'''
        while self.live:
            newY = np.zeros_like(self.xx) #this will eventually be a call to the spectrometer asking for data
            for _ in range(self.noOfAv): #the call to the spectrometer will be inside an averaging loop
                newY += noisyGaussian(self.xx) #for now, we simulate averaging
            newY /= self.noOfAv
            
            self.data.set_ydata(newY) #write the new data to the plot
            
            with self.plot_output: #output the new data
                self.fig.canvas.draw_idle() 
            
            for _ in range(int(self.exposureTime / 10)): #simulate the time it takes 
                if not self.live: break
                time.sleep(0.01)

    def onExposureUpdate(self, change):
        '''Writes the new exposure to a variable. We do not want to use .value in case the thread gets confused.'''
        self.exposureTime = change['new']
    
    def onAvUpdate(self, change):
        '''Writes the new average to a variable. We do not want to use .value in case the thread gets confused.'''
        self.noOfAv = change['new']

    def onFigSaveRequested(self, button_object):
        ''' Saves the current figure. If this is the first figure saved in this run, requests the user to select a folder to save to.'''
        with self.text_output:
            self.text_output.clear_output()
            print('Still a WIP!')

In [4]:
PL_GUI(xx)

/var/folders/m7/931qdrln5nn0dz8vsc5s20nh0000gn/T/ipykernel_42913/585037901.py:93: RuntimeWarning: invalid value encountered in divide
  newY /= self.noOfAv
/var/folders/m7/931qdrln5nn0dz8vsc5s20nh0000gn/T/ipykernel_42913/585037901.py:93: RuntimeWarning: invalid value encountered in divide
  newY /= self.noOfAv
/var/folders/m7/931qdrln5nn0dz8vsc5s20nh0000gn/T/ipykernel_42913/585037901.py:93: RuntimeWarning: invalid value encountered in divide
  newY /= self.noOfAv
/var/folders/m7/931qdrln5nn0dz8vsc5s20nh0000gn/T/ipykernel_42913/585037901.py:93: RuntimeWarning: invalid value encountered in divide
  newY /= self.noOfAv
/var/folders/m7/931qdrln5nn0dz8vsc5s20nh0000gn/T/ipykernel_42913/585037901.py:93: RuntimeWarning: invalid value encountered in divide
  newY /= self.noOfAv
